# Gold Layer

# Load Silver Tables

In [0]:
silver_customers_df=spark.read.table("zomato_project.silver.silver_customers")
silver_orders_df=spark.read.table("zomato_project.silver.silver_orders")
silver_deliveries_df=spark.read.table("zomato_project.silver.silver_deliveries")
silver_delivery_persons_df=spark.read.table("zomato_project.silver.silver_delivery_persons")
silver_restaurants_df=spark.read.table("zomato_project.silver.silver_restaurants")

silver_customers_df.display()
silver_orders_df.display()
silver_deliveries_df.display()
silver_delivery_persons_df.display()
silver_restaurants_df.display()

In [0]:
from pyspark.sql.functions import col

gold_df = silver_orders_df.alias("o") \
    .join(silver_customers_df.alias("c"),
          col("o.customer_id") == col("c.customer_id"),
          "left") \
    .join(silver_deliveries_df.alias("d"),
          col("o.order_id") == col("d.order_id"),
          "left")

gold_df.display()

In [0]:
from pyspark.sql.functions import sum,count
from pyspark.sql import functions as F

### Total Revenue by City

In [0]:
revenue_city = gold_df.groupBy("location") \
    .agg(F.sum("total_amount").alias("total_revenue"))
display(revenue_city)

### Top Restaurants by Orders

In [0]:
top_restaurants = gold_df.join(silver_restaurants_df, "restaurant_id").groupBy("restaurant_id") \
    .agg(count("o.order_id").alias("total_orders")) \
    .orderBy("total_orders", ascending=False).limit(10)
top_restaurants.display()

### Customer Segmentation (High-value customers)

In [0]:
customer_value = gold_df.groupBy("name") \
    .agg(sum("total_amount").alias("total_spent"))

high_value_customers = customer_value.filter("total_spent > 5000")
display(high_value_customers)

### Top Cities by Orders

In [0]:
from pyspark.sql.functions import desc
top_cities = gold_df.groupBy("location") \
    .agg(count("o.order_id").alias("total_orders")) \
    .orderBy("total_orders",ascending=False).limit(10).offset(2)
display(top_cities)

### Repeat Customer Analysis

In [0]:
repeat_customers = gold_df.groupBy("name") \
    .agg(count("o.order_id").alias("order_count")) \
    .filter("order_count > 1")
display(repeat_customers)

### Kpis

In [0]:
from pyspark.sql.functions import count, sum, avg

kpi_df = gold_df.agg(
    count("o.order_id").alias("total_orders"),
    sum("total_amount").alias("total_revenue"),
    avg("d.delivery_time").alias("avg_delivery_time")
)
display(kpi_df)

In [0]:
from pyspark.sql.functions import col
gold_df1= silver_restaurants_df.alias("r") \
    .join(silver_orders_df.alias("o"),
          col("r.restaurant_id") == col("o.restaurant_id"),
          "left") \
    .join(silver_deliveries_df.alias("d"),
          col("o.order_id") == col("d.order_id"),
          "left") 
gold_df1.display()

### Revenue by Restaurant

In [0]:
gold_df1 = gold_df1.withColumnRenamed("name", "restaurant_name")
revenue_by_restaurant = gold_df1.groupBy("restaurant_name") \
    .agg(F.sum("total_amount").alias("total_revenue")) \
    .orderBy(F.desc("total_revenue")).limit(10)

display(revenue_by_restaurant)

### Orders per Restaurant

In [0]:
orders_per_restaurant = gold_df1.groupBy("restaurant_name") \
    .agg(F.count("o.order_id").alias("total_orders")) \
    .orderBy(F.desc("total_orders"))
display(orders_per_restaurant)

### Average Order Value per Restaurant

In [0]:
avg_order_value = gold_df1.groupBy("restaurant_name") \
    .agg(F.avg("total_amount").alias("avg_order_value")).limit(5)
display(avg_order_value)

### Average Delivery Time per Restaurant

In [0]:
avg_delivery_time_rest = gold_df1.groupBy("restaurant_name") \
    .agg(F.avg("d.delivery_time").alias("avg_delivery_time"))
display(avg_delivery_time_rest)

### High Revenue Restaurants with Fast Delivery

In [0]:
high_perf_restaurants = gold_df1.groupBy("restaurant_name") \
    .agg(
        F.sum("total_amount").alias("revenue"),
        F.avg("d.delivery_time").alias("avg_time")
    ).filter("revenue > 5000 AND avg_time < 30")
display(high_perf_restaurants)

In [0]:
revenue_city.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.revenue_by_city")
top_restaurants.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.top_restaurants")
customer_value.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.customer_segmentation")
kpi_df.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.kpi_dashboard")

In [0]:
revenue_by_restaurant.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.top_restaurants_revenue")
avg_delivery_time_rest.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.avg_delivery_time_rest")
high_perf_restaurants.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.high_perf_restaurants")

In [0]:
top_cities.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.city_orders")
repeat_customers.write.format("delta").mode("overwrite").saveAsTable("zomato_project.gold.repeat_customers")